# R₀ Calculation for SEIRS-SEI Model

Deriving the basic reproduction number $R_0$ using the Next Generation Matrix (NGM) approach.

## 1. Model Equations (as in Test_SEIRS_SEI_Model.ipynb)

### Human (SEIRS):
$$\frac{dS_H}{dt} = r_H N - a b_2 \frac{I_M(t)}{N} S_H + \omega R_H$$
$$\frac{dE_H}{dt} = a b_2 \frac{I_M(t)}{N} S_H - a b_2 \frac{I_M(t-\tau_H)}{N} S_H(t-\tau_H)$$
$$\frac{dI_H}{dt} = a b_2 \frac{I_M(t-\tau_H)}{N} S_H(t-\tau_H) - \gamma I_H$$
$$\frac{dR_H}{dt} = \gamma I_H - \omega R_H$$

### Mosquito (SEI):
$$\frac{dS_M}{dt} = \Lambda - a b_1 \frac{I_H(t)}{N} S_M - \mu S_M$$
$$\frac{dE_M}{dt} = a b_1 \frac{I_H(t)}{N} S_M - a b_1 \frac{I_H(t-\tau_M)}{N} S_M(t-\tau_M) \cdot \ell - \mu E_M$$
$$\frac{dI_M}{dt} = a b_1 \frac{I_H(t-\tau_M)}{N} S_M(t-\tau_M) \cdot \ell - \mu I_M$$

where $\ell = e^{-\mu \tau_M}$ is the probability of surviving the sporogonic delay.

## 2. Next Generation Matrix Approach

**Infected compartments:** $[E_H, I_H, E_M, I_M]$

Define:
- $\mathcal{F}$: rate of new infections
- $\mathcal{V} = \mathcal{V}^{-} - \mathcal{V}^{+}$: rate of transitions out minus in

At the **Disease-Free Equilibrium (DFE)**:
- $S_H^* = N, E_H^* = I_H^* = R_H^* = 0$
- $S_M^* = M, E_M^* = I_M^* = 0$

Then $R_0 = \rho(FV^{-1})$ where $\rho$ is the spectral radius.

## 3. Computing $\mathcal{F}$ and $\mathcal{V}$

### $\mathcal{F}$ (New Infections)

Only terms that create **newly infected** individuals:

- $\mathcal{F}_1 = a b_2 \frac{I_M}{N} S_H$ → at DFE: $a b_2 I_M$
- $\mathcal{F}_2 = 0$
- $\mathcal{F}_3 = a b_1 \frac{I_H}{N} S_M$ → at DFE: $a b_1 \frac{M}{N} I_H$
- $\mathcal{F}_4 = 0$

### $\mathcal{V} = \mathcal{V}^{-} - \mathcal{V}^{+}$

**For $E_H$:**
- $\mathcal{V}_1^{-} = \sigma_H E_H$ (progression to $I_H$)
- $\mathcal{V}_1^{+} = 0$
- $\mathcal{V}_1 = \sigma_H E_H$

**For $I_H$:**
- $\mathcal{V}_2^{-} = \gamma I_H$ (recovery)
- $\mathcal{V}_2^{+} = \sigma_H E_H$ (inflow from $E_H$)
- $\mathcal{V}_2 = \gamma I_H - \sigma_H E_H$

**For $E_M$:**
- $\mathcal{V}_3^{-} = \sigma_M E_M + \mu E_M$ (progression to $I_M$ + death)
- $\mathcal{V}_3^{+} = 0$
- $\mathcal{V}_3 = (\sigma_M + \mu) E_M$

**For $I_M$:**
- $\mathcal{V}_4^{-} = \mu I_M$ (death)
- $\mathcal{V}_4^{+} = \sigma_M E_M$ (inflow from $E_M$)
- $\mathcal{V}_4 = \mu I_M - \sigma_M E_M$

## 4. Jacobians at DFE

### F Jacobian (∂ℱᵢ/∂xⱼ at DFE)

State vector order: $[E_H, I_H, E_M, I_M]$

$$F = \begin{pmatrix} 
0 & 0 & 0 & a b_2 \\
0 & 0 & 0 & 0 \\
0 & a b_1 \frac{M}{N} & 0 & 0 \\
0 & 0 & 0 & 0
\end{pmatrix}$$

### V Jacobian (∂𝒱ᵢ/∂xⱼ at DFE)

$$V = \begin{pmatrix} 
\sigma_H & 0 & 0 & 0 \\
-\sigma_H & \gamma & 0 & 0 \\
0 & 0 & \sigma_M + \mu & 0 \\
0 & 0 & -\sigma_M & \mu
\end{pmatrix}$$

This V matrix is **block diagonal** with two 2×2 blocks. Both blocks have non-zero determinant:

- Human block: $\det = \sigma_H \cdot \gamma > 0$
- Mosquito block: $\det = \mu(\sigma_M + \mu) > 0$

Therefore $\det(V) = \sigma_H \cdot \gamma \cdot \mu \cdot (\sigma_M + \mu) > 0$, and V is **invertible**.

## 5. Next Generation Matrix

$K = F V^{-1}$

**Human block inversion:**
$$V_h = \begin{pmatrix} \sigma_H & 0 \\ -\sigma_H & \gamma \end{pmatrix}, \quad V_h^{-1} = \begin{pmatrix} 1/\sigma_H & 0 \\ 1/\gamma & 1/\gamma \end{pmatrix}$$

**Mosquito block inversion:**
$$V_m = \begin{pmatrix} \sigma_M+\mu & 0 \\ -\sigma_M & \mu \end{pmatrix}, \quad V_m^{-1} = \begin{pmatrix} 1/(\sigma_M+\mu) & 0 \\ \sigma_M/(\mu(\sigma_M+\mu)) & 1/\mu \end{pmatrix}$$

**NGM:**

$$K = F V^{-1} = \begin{pmatrix} 
0 & 0 & \frac{a b_2 \sigma_M}{\mu(\sigma_M+\mu)} & \frac{a b_2}{\mu} \\
0 & 0 & 0 & 0 \\
\frac{a b_1 M}{N \gamma} & \frac{a b_1 M}{N \gamma} & 0 & 0 \\
0 & 0 & 0 & 0
\end{pmatrix}$$

The characteristic polynomial is $\lambda^4 - \frac{a^2 b_1 b_2 M}{N \gamma} \cdot \frac{\sigma_M}{\mu(\sigma_M+\mu)} \lambda^2 = 0$

Eigenvalues: $0, 0, \pm \sqrt{\frac{a^2 b_1 b_2 M}{N \gamma} \cdot \frac{\sigma_M}{\mu(\sigma_M+\mu)}}$

## 6. Final R₀ Formula

The spectral radius (dominant eigenvalue) is:

$$R_0 = \sqrt{ \frac{a^2 b_1 b_2 M}{N \gamma} \cdot \frac{\sigma_M}{\mu(\sigma_M+\mu)} } = \sqrt{ \frac{a^2 b_1 b_2 M \ell}{N \gamma \mu} }$$

where $\ell = \frac{\sigma_M}{\sigma_M+\mu} = e^{-\mu \tau_M}$ is the probability of surviving the sporogonic delay.

Equivalently:

$$R_0 = \sqrt{R_0^{human} \cdot R_0^{mosquito}}$$

where:
- $R_0^{human} = \frac{a b_2}{\gamma}$ (human side)
- $R_0^{mosquito} = \frac{a b_1 M}{N} \cdot \frac{\ell}{\mu}$ (mosquito side)

## Summary

| Component | Formula |
|-----------|---------|
**Human (SEIRS)** | $R_0^{human} = \frac{a b_2}{\gamma}$ |
**Mosquito (SEI)** | $R_0^{mosquito} = \frac{a b_1 M}{N} \cdot \frac{e^{-\mu \tau_M}}{\mu}$ |
**Combined** | $R_0 = \sqrt{R_0^{human} \cdot R_0^{mosquito}}$ |

**Key insights:**
- $\tau_H$ cancels out — no mortality during human incubation
- $\tau_M$ appears via $\ell = e^{-\mu \tau_M}$ — mosquito mortality during sporogony reduces transmission